# Spatially Masked Regression on AJILE12: mask sweep and electrode coverage

Ostadsharif Memar & Dehghani (2026), arXiv:2606.11415, on DANDI:000055 version 0.220127.0436
(the only published version; see `reproduce_memar_2026.ipynb` for the header, the version choice
and the full data check). Claims tested here: fig4a-mask-sweep-ieeg and fig5a-coverage-ieeg.
Settings and deviations as in `reproduce_memar_2026.ipynb` (every 10th sample for the gradient
steps, 200 epochs).

In [ ]:
import os, time, zlib, json, resource
import numpy as np
import pandas as pd
from scipy import stats
import smr

T0 = time.time()
PAPER = "memar-2026"
def seed(claim):
    return zlib.crc32(f"{PAPER}/{claim}".encode())

DECIM, MAX_EPOCH = 10, 200
SUBJECTS = [f"sub-{i:02d}" for i in range(1, 13)]
if os.environ.get("SMR_TEST"):                 # development only: two subjects
    SUBJECTS = ["sub-07", "sub-04"]
IDX = [int(s[-2:]) - 1 for s in SUBJECTS]
print("dataset", smr.DANDISET, smr.VERSION)

## Data check

The trials of every subject, preprocessed as in `reproduce_memar_2026.ipynb` (reading a only; its data check covers the metadata, duplicates and the clock).

In [ ]:
t = time.time()
DATA = {}
for s in SUBJECTS:
    S = smr.load_subject(s, readings=("a",))
    DATA[s] = smr.drop_uncoordinated(S)
    x = DATA[s]["a"]
    print(s, "sessions", len(S["sessions"]), "trials", x.shape[0], "electrodes", x.shape[1], "samples", x.shape[2],
          "dropped without coordinates", DATA[s]["n_dropped_el"], f"range {x.min():.2f} to {x.max():.2f}, SD {x.std():.3f}",
          f"({time.time() - t:.0f} s)", flush=True)
print("trials in all", sum(S["a"].shape[0] for S in DATA.values()))
print("peak memory MB", resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e3)

In [ ]:
def run_all(m, rng, reading="a", **kw):
    # one SMR condition on every subject, in order, from one generator
    out = {}
    for s in SUBJECTS:
        S = DATA[s]
        out[s] = smr.run_smr(S[reading].astype(np.float32), smr.neighbours(S["xyz"]), m, rng,
                             decim=DECIM, max_epoch=MAX_EPOCH, **kw)
    return out

def subj(res):
    return np.array([res[s]["dc"].mean() for s in SUBJECTS])

def msd(v):
    return f"{np.mean(v):.3f} ± {np.std(v):.3f}"

## fig4a-mask-sweep-ieeg

> In iEEG, DistCorr decreases as more of the target's local neighborhood is masked (mask
> intensity 0, 25, 50, 75 and 100%).

ceil(m × 9) neighbours drawn at random per electrode (Results III D), m in 0, 0.25, 0.5, 0.75 and 1.0, one generator for the whole sweep. The code's rule, round(m × 9), differs at m = 0.25 (2 in place of 3) and m = 0.5 (4 in place of 5), and is shown as a variant.

In [ ]:
t = time.time()
rng_sweep = np.random.default_rng(seed("fig4a-mask-sweep-ieeg"))
SWEEP = {}
for m in (0.0, 0.25, 0.5, 0.75, 1.0):
    SWEEP[m] = run_all(m, rng_sweep, "a")
    print(f"m = {m} done ({time.time() - t:.0f} s)", flush=True)
ms = sorted(SWEEP)
sw = np.array([subj(SWEEP[m]) for m in ms])          # (m, subject)
print(pd.DataFrame(sw.T.round(3), index=SUBJECTS, columns=[f"m={m}" for m in ms]).to_string())
avg = sw.mean(1)
print("average over subjects:", dict(zip(ms, avg.round(3))))
print("masked neighbours per electrode:", {m: int(np.ceil(m * 9 - 1e-9)) for m in ms})
steps = np.diff(avg)
print("average decreases at every step:", bool((steps < 0).all()), "steps", steps.round(4).tolist())
mono = (np.diff(sw, axis=0) < 0).all(0)
print("subjects decreasing at every step:", int(mono.sum()), "of 12", [s for s, k in zip(SUBJECTS, mono) if not k])
print("m = 1.0 below m = 0:", avg[-1] < avg[0], f"({avg[-1]:.3f} against {avg[0]:.3f})")
print("m = 1.0 values for full-mask-above-zero-ieeg:", dict(zip(SUBJECTS, sw[-1].round(3))), "average", round(avg[-1], 3))

Variant: the code's round(m × 9) at m = 0.25 and 0.5 (a second generator for the claim).

In [ ]:
rng_round = np.random.default_rng(seed("fig4a-mask-sweep-ieeg") + 1)
ROUND = {m: subj(run_all(m, rng_round, "a", rule="round")) for m in (0.25, 0.5)}
alt = [sw[0].mean(), ROUND[0.25].mean(), ROUND[0.5].mean(), sw[3].mean(), sw[4].mean()]
print("averages with round at 0.25 and 0.5:", np.round(alt, 3).tolist(), "; decreasing at every step:",
      bool((np.diff(alt) < 0).all()))
print("subjects decreasing at every step with round:",
      int((np.diff(np.array([sw[0], ROUND[0.25], ROUND[0.5], sw[3], sw[4]]), axis=0) < 0).all(0).sum()), "of 12")

**Result: reproduced.** The subject average falls at every step, 0.712, 0.674, 0.650, 0.628 and 0.602 from m = 0 to m = 1.0, and in each of the 12 subjects; with the code's round rule it also falls at every step (0.712, 0.685, 0.660, 0.628, 0.602).

## fig5a-coverage-ieeg

> In iEEG, reconstruction is best with all electrodes as predictors, and local electrodes alone
> consistently outperform non-local electrodes alone.

All = only the target masked (m = 0 of the sweep); Non-local = target and neighbourhood masked
(m = 1.0 of the sweep); Local = only the 9 neighbours as predictors (Results III E; run_coverage_experiment.py).

In [ ]:
t = time.time()
LOCAL = run_all(0.0, np.random.default_rng(seed("fig5a-coverage-ieeg")), "a", mode="local")
print(f"local done ({time.time() - t:.0f} s)")
cov = pd.DataFrame({"subject": SUBJECTS, "all": subj(SWEEP[0.0]), "local": subj(LOCAL), "non_local": subj(SWEEP[1.0])})
print(cov.round(3).to_string(index=False))
a_, l_, n_ = cov["all"].mean(), cov["local"].mean(), cov["non_local"].mean()
print(f"averages: all {a_:.3f}, local {l_:.3f}, non-local {n_:.3f}; ordered all > local > non-local: {a_ > l_ > n_}")
print("subjects with local above non-local:", int((cov.local > cov.non_local).sum()), "of 12;",
      "with all above local:", int((cov["all"] > cov.local).sum()), "of 12")

**Result: not reproduced.** All (0.712) is highest, but Local alone (0.567) is below Non-local (0.602) on average and in 10 of 12 subjects, so the order All > Local > Non-local does not hold.

## Summary

| Claim | Outcome |
|---|---|
| fig4a-mask-sweep-ieeg | reproduced (0.712 to 0.602, falling at every step in 12 of 12 subjects) |
| fig5a-coverage-ieeg | not reproduced (Local 0.567 below Non-local 0.602) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method. The differences and the dataset findings
are listed in the summary of `reproduce_memar_2026.ipynb`.


In [ ]:
print(f"notebook ran {(time.time() - T0) / 60:.1f} min; peak memory "
      f"{resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.1f} GB")